# 05 Multi-Label Failure Mode Training & Evaluation

This notebook trains and evaluates Multi-Label Classification models to predict 5 specific machine failure modes (`twf`, `hdf`, `pwf`, `osf`, `rnf`).

## 1. Import Libraries & Setup

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_train shape: (8000, 9) | X_test shape: (2000, 9)
Failure mode counts in Training set:
twf    36
hdf    86
pwf    82
osf    82
rnf    15

Failure mode counts in Testing set:
twf    10
hdf    29
pwf    13
osf    16
rnf     4


## 2. Load Processed Datasets & Target Formulation

In [2]:
data_dir = '../data/processed'
if not os.path.exists(data_dir):
    data_dir = 'data/processed'

X_train = pd.read_csv(os.path.join(data_dir, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(data_dir, 'X_test.csv'))
y_train_full = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
y_test_full  = pd.read_csv(os.path.join(data_dir, 'y_test.csv'))

failure_cols = ['twf', 'hdf', 'pwf', 'osf', 'rnf']
y_train = y_train_full[failure_cols]
y_test  = y_test_full[failure_cols]

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print("Failure mode counts in Training set:")
print(y_train.sum())
print("\nFailure mode counts in Testing set:")
print(y_test.sum())

Multi-Label Classification Model trained successfully.


## 3. Feature Scaling & Multi-Label Model Training

In [3]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),       columns=X_test.columns)

lgbm_base = LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE, verbose=-1)
model_multi = MultiOutputClassifier(lgbm_base)
model_multi.fit(X_train_scaled, y_train)
print("Multi-Label Classification Model trained successfully.")

              Recall  Precision  F1-Score
Failure Mode                             
TWF           0.0000     0.0000    0.0000
HDF           1.0000     1.0000    1.0000
PWF           0.9231     0.9231    0.9231
OSF           1.0000     0.9412    0.9697
RNF           0.0000     0.0000    0.0000


## 4. Evaluation per Failure Mode on Test Set

In [4]:
y_pred_multi = model_multi.predict(X_test_scaled)
y_pred_df = pd.DataFrame(y_pred_multi, columns=failure_cols)

eval_rows = []
for col in failure_cols:
    eval_rows.append({
        'Failure Mode': col.upper(),
        'Recall': round(recall_score(y_test[col], y_pred_df[col], zero_division=0), 4),
        'Precision': round(precision_score(y_test[col], y_pred_df[col], zero_division=0), 4),
        'F1-Score': round(f1_score(y_test[col], y_pred_df[col], zero_division=0), 4)
    })

eval_df = pd.DataFrame(eval_rows).set_index('Failure Mode')
display(eval_df)

Multi-label model and scaler saved to c:\Users\VICTUS\OneDrive\Documents\Edwin's_Project\CompasFeast\smart_manufacturing_maintenance\models/


## 5. Model Export

In [5]:
joblib.dump(model_multi, os.path.join(model_dir, 'multi_label_model.pkl'))
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))
print(f"Multi-label model and scaler saved to {model_dir}/")